In [ ]:
# setup class hierarchy and child and parents map
import numpy as np
from pathlib import Path
import json
import csv
import torch
import pandas as pd

np.random.seed(42)
torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
ROOT = Path("Amazon_products")

CLASS_HIER_PATH = ROOT / "class_hierarchy.txt"
parents_map = {i: [] for i in range(531)}
children_map = {i: [] for i in range(531)}

with open(CLASS_HIER_PATH, "r") as f:
    for line in f:
        p, c = map(int, line.strip().split("\t"))
        parents_map[c].append(p)
        children_map[p].append(c)


def hierarchy_respect_ratio_from_labels(csv_path, parents_map, n_classes=531):
    """Measure how often a child class is predicted when its parent is not."""
    df = pd.read_csv(csv_path)

    # binary prediction matrix [N, C]
    Y = torch.zeros((len(df), n_classes), dtype=torch.int)

    for i, labels in enumerate(df["label"]):
        if isinstance(labels, str) and labels != "":
            for l in map(int, labels.split(",")):
                Y[i, l] = 1

    total, ok = 0, 0
    for child, parents in parents_map.items():
        for p in parents:
            child_pred = Y[:, child]
            parent_pred = Y[:, p]

            # valid if child not predicted OR parent predicted
            ok += ((child_pred == 0) | (parent_pred == 1)).sum().item()
            total += Y.size(0)

    return ok / total


def json_to_csv_labels(json_path, csv_path, start_idx=29487):
    """Convert JSON predictions to CSV format."""
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    with open(csv_path, "w", encoding="utf-8") as f:
        f.write("id,label\n")

        for i in range(start_idx, len(data)):
            entry = data[str(i)]
            labels = entry["labels"]
            label_str = ",".join(str(x) for x in labels)
            f.write(f'{i - start_idx},"{label_str}"\n')
            
    print(f"CSV saved to {csv_path}")

json_to_csv_labels(
    json_path="SilverGeneration/SilverRemake/silver_train_mpnet.json",
    csv_path="Submission/predictions_noshot.csv",
    start_idx=29487
)

ratio = hierarchy_respect_ratio_from_labels(
    csv_path="Submission/predictions_noshot.csv",
    parents_map=parents_map,
    n_classes=531
)

print(f"Hierarchy respect ratio (discrete): {ratio:.4f}")

CSV saved to Submission/predictions_noshot.csv
Hierarchy respect ratio (discrete): 0.9995
